In [22]:
import pandas as pd
import numpy as np
import json
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [23]:
input_dir = "../../data/processed/feature_selection"

with open(f"{input_dir}/selected_features.json") as f:
    meta = json.load(f)

target = meta["target"]
selected_features = meta["selected_features"]

train_final = pd.read_csv(f"{input_dir}/train_final.csv")
validation_final = pd.read_csv(f"{input_dir}/validation_final.csv")
test_final = pd.read_csv(f"{input_dir}/test_final.csv")

X_train_final = train_final[selected_features]
y_train = train_final[target]

X_validation_final = validation_final[selected_features]
y_validation = validation_final[target]

X_test_final = test_final[selected_features]
y_test = test_final[target]

print("Number of features:", len(selected_features))
print("Train:", X_train_final.shape)
print("Validation:", X_validation_final.shape)
print("Test:", X_test_final.shape)

Number of features: 26
Train: (16200, 26)
Validation: (2760, 26)
Test: (2550, 26)


In [24]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

xgb_model.fit(X_train_final, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [25]:
y_xgb_pred = xgb_model.predict(X_validation_final)

mae_xgb = mean_absolute_error(y_validation, y_xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_validation, y_xgb_pred))
r2_xgb = r2_score(y_validation, y_xgb_pred)

print("XGBoost Validation Metrics")
print(f"MAE:  {mae_xgb:.4f}")
print(f"RMSE: {rmse_xgb:.4f}")
print(f"R²:   {r2_xgb:.4f}")

XGBoost Validation Metrics
MAE:  112.1912
RMSE: 149.3852
R²:   0.9275


## Tuning 

In [26]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 6, 8],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [ ]:
xgb_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

random_search.fit(X_train_final, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.


In [ ]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV RMSE:")
print(-random_search.best_score_)

Best Parameters:
{'subsample': 1.0, 'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

Best CV RMSE:
482.81818680556927


In [ ]:
best_xgb = random_search.best_estimator_

y_xgb_tuned_pred = best_xgb.predict(X_validation_final)

mae_xgb_tuned = mean_absolute_error(y_validation, y_xgb_tuned_pred)
rmse_xgb_tuned = np.sqrt(mean_squared_error(y_validation, y_xgb_tuned_pred))
r2_xgb_tuned = r2_score(y_validation, y_xgb_tuned_pred)

print("Tuned XGBoost Validation Metrics")
print(f"MAE:  {mae_xgb_tuned:.4f}")
print(f"RMSE: {rmse_xgb_tuned:.4f}")
print(f"R²:   {r2_xgb_tuned:.4f}")

Tuned XGBoost Validation Metrics
MAE:  308.9664
RMSE: 403.0915
R²:   0.9894


In [ ]:
y_xgb_test_pred = best_xgb.predict(X_test_final)

mae_xgb_test = mean_absolute_error(y_test, y_xgb_test_pred)
rmse_xgb_test = np.sqrt(mean_squared_error(y_test, y_xgb_test_pred))
r2_xgb_test = r2_score(y_test, y_xgb_test_pred)

print("Tuned XGBoost Test Metrics")
print(f"MAE:  {mae_xgb_test:.4f}")
print(f"RMSE: {rmse_xgb_test:.4f}")
print(f"R²:   {r2_xgb_test:.4f}")

Tuned XGBoost Test Metrics
MAE:  446.1986
RMSE: 744.4561
R²:   0.9769
